# Forward test: the live strategy and 34 paper strategies (from Fri Oct 2, 2026)

Replaces the archived backtest (`Archive/backtest.ipynb`, reference only). It counts only what happens from the `backtest_engine.FORWARD_START` close on: the real Alpaca account and 35 paper-only strategies (`forward_test.STRATEGIES`, never traded) against QQQ and SPY. After 12 weeks the best one (ranked by `forward_test.RANK_RULE`, chosen before any result) may replace the live rules.

- **History:** `Reports/forward_test_daily.csv` (the account) and `Reports/forward_strategies.csv` + `Reports/forward_strategies_holdings.csv` (each strategy's value and holdings), one row per trading day, saved by the launchd job `com.stockanalysis.forwardtest` (Mon-Fri 4:15 PM CT, `python forward_test.py --record`; read-only GET requests, no orders).
- **Strategy line:** account equity minus deposits made after the first saved day (deposits are not returns). QQQ / SPY: daily closes from `Reports/benchmark_prices.csv`, comparison only (never traded).
- **Cost:** fills vs the decision price (the stock's close on the order's As_Of day in `signal_analysis.csv`, the 2:30 PM bar on decision days); + = it cost money.

This notebook only adds the paper strategies' missing days (saved data, no requests); it runs no pipeline step and places no orders. Set `READ_FILLS = False` to skip the Alpaca read (then only the saved rows are used).

In [ ]:
import pandas as pd

import backtest_engine as be
import forward_test as ft

READ_FILLS = True        # read the account's fills (GET only) for the per-pick table below
daily = pd.read_csv(ft.DAILY_CSV, parse_dates=["Date"]) if __import__("os").path.exists(ft.DAILY_CSV) else None
print(f"Forward test from {pd.Timestamp(be.FORWARD_START):%a %b %-d, %Y}:", "no saved rows yet" if daily is None 
      else f"{len(daily)} saved day(s), last {daily['Date'].max():%a %b %-d}")

## Leaderboard
Every strategy + your account + QQQ / SPY since the start close: total return, median weekly return (Friday to Friday), max drawdown and finished weeks, sorted by the ranking rule printed below. `python forward_test.py` adds any missing days first (saved data + one free daily-bar download for volume; no account request, no orders).

In [ ]:
ft.update_strategies()
board = ft.leaderboard(daily)
print(ft.verdict(board), ft.RANK_RULE)
board.round(2)

## Each strategy: rule and holdings now
Target weights on the latest saved day (paper only).

In [ ]:
held = ft.holdings()
pd.DataFrame([{"Strategy": c["name"], "Rule": c["rule"], "Holdings now": held.get(c["name"], "cash")} for c in ft.STRATEGIES])

## Picks and trading cost
A pick = a buy into a flat position on/after the start until the position is back to 0 (positions held before the start are not counted). Win rate = closed picks with a profit.

In [ ]:
if daily is not None and len(daily):
    last = daily.iloc[-1]
    n, w = int(last["Closed_Picks"]), int(last["Winning_Picks"])
    traded, cost = float(last["Traded_USD"]), float(last["Cost_USD"])
    print(f"Closed picks {n}" + (f", win rate {w / n:.0%}" if n else " (no closed picks yet)")
          + f" | traded ${traded:,.0f}, cost vs decision price ${cost:,.2f}"
          + (f" ({cost / traded * 1e4:+.1f} bps)" if traded else ""))
if READ_FILLS:
    import alpaca_paper as ap
    picks = ft.picks(ap.PaperAccount().fills())
    display(picks.round(2) if len(picks) else "No closed picks since the start yet.")